# Flow-matching doodles: cats, fish and butterflies

**Goal:** train a small flow-matching model from scratch that turns Gaussian noise into 28 × 28 doodles of a cat, a fish or a butterfly, and test two questions about it.

**Data:** a fixed random subset of Google's Quick, Draw! dataset (CC BY 4.0).

**Two questions tested**

1. *Training factor:* how do the generated drawings change between early and later training checkpoints?
2. *Inference factor:* how many Euler steps does the sampler need before the drawings stop improving?

**How to run on Colab:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Everything is saved to the folder `partC_outputs/` and zipped at the end.

---
## 1. Setup

All settings are in one place. `RUN_TRAINING = False` skips training and loads the saved weights instead (the reproduction route).

In [ ]:
import os, json, math, time, shutil, platform, urllib.request, urllib.parse

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

RUN_TRAINING = True                # False = load saved weights from partC_outputs/weights/

CLASSES = ["cat", "fish", "butterfly"]
N_PER_CLASS = 6000                 # drawings taken per category
N_TRAIN_PER_CLASS = 5000           # the remaining 1,000 per category are held out
SEED = 6010

BATCH_SIZE = 128
LEARNING_RATE = 3e-4
TOTAL_STEPS = 6000                 # training steps (one step = one batch)
CHECKPOINT_STEPS = [100, 500, 2000, 6000]   # four points during training; the last is the final model

BASE_ODE_STEPS = 50                # Euler steps used by the baseline sampler
ODE_STEPS_LIST = [2, 5, 10, 20, 50]

EVAL_SEEDS = list(range(100))      # predetermined seed list: 100 images per class are scored
N_GRID = 16                        # the first 16 seeds per class are shown in the grids

DATA_DIR = "data"
OUT_DIR = "partC_outputs"
for folder in [DATA_DIR, OUT_DIR, f"{OUT_DIR}/weights", f"{OUT_DIR}/grids", f"{OUT_DIR}/figures"]:
    os.makedirs(folder, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
hardware = torch.cuda.get_device_name(0) if device == "cuda" else (platform.processor() or "CPU")
versions = {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__,
            "pandas": pd.__version__, "matplotlib": matplotlib.__version__}
print("Device:", device, "|", hardware)
print(versions)

# Colours and plot style used by every chart
BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#52514e"
CLASS_COLOURS = [BLUE, ORANGE, AQUA]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "grid.linewidth": 0.8, "axes.axisbelow": True,
                     "lines.linewidth": 2, "lines.markersize": 7, "font.size": 10,
                     "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight"})

---
## 2. Data

**Source.** One `.npy` file per category from the official Quick, Draw! bucket. Each row is one 28 × 28 grayscale drawing stored as 784 numbers from 0 to 255.

**Subset.** 6,000 drawings per category, picked at random with a fixed seed. Nothing was filtered or hand-picked, so odd or ambiguous drawings stay in.

**Split.** The split is made on the original drawings: 5,000 per category for training and 1,000 held out. No augmentation is used, so the counts below are real drawings. The held-out drawings are never used to train the flow model.

**Preprocessing.** Pixel values are rescaled from [0, 255] to [−1, 1]. The image size stays 28 × 28.

The row number of every drawing used is saved to `split_manifest.csv`.

In [ ]:
BASE_URL = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/"

def to_tensor(rows):
    # (N, 784) uint8 in [0, 255]  ->  (N, 1, 28, 28) float in [-1, 1]
    return torch.from_numpy(rows.astype(np.float32) / 127.5 - 1.0).reshape(-1, 1, 28, 28)

rng = np.random.default_rng(SEED)
train_parts, test_parts, manifest = [], [], []
for label, name in enumerate(CLASSES):
    path = f"{DATA_DIR}/{name}.npy"
    if not os.path.exists(path):
        urllib.request.urlretrieve(BASE_URL + urllib.parse.quote(name) + ".npy", path)
    full = np.load(path)
    chosen = rng.choice(len(full), N_PER_CLASS, replace=False)       # fixed random subset
    train_rows, test_rows = chosen[:N_TRAIN_PER_CLASS], chosen[N_TRAIN_PER_CLASS:]
    train_parts.append((to_tensor(full[train_rows]), torch.full((len(train_rows),), label)))
    test_parts.append((to_tensor(full[test_rows]), torch.full((len(test_rows),), label)))
    manifest += [(name, int(r), "train") for r in train_rows] + [(name, int(r), "held_out") for r in test_rows]
    print(f"{name:10s} file has {len(full):,} drawings -> {len(train_rows):,} train, {len(test_rows):,} held out")
    del full

train_x = torch.cat([x for x, _ in train_parts]).to(device)
train_y = torch.cat([y for _, y in train_parts]).to(device)
test_x = torch.cat([x for x, _ in test_parts]).to(device)
test_y = torch.cat([y for _, y in test_parts]).to(device)

manifest = pd.DataFrame(manifest, columns=["class", "original_row", "split"])
manifest.to_csv(f"{OUT_DIR}/split_manifest.csv", index=False)
print("train:", tuple(train_x.shape), " held out:", tuple(test_x.shape),
      " value range:", train_x.min().item(), "to", train_x.max().item())

In [ ]:
def grid_array(images, n_cols, pad=2):
    # Tile images of shape (N, 1, 28, 28) into one big picture with n_cols images per row.
    images = images.detach().cpu().numpy()[:, 0]
    n_rows = math.ceil(len(images) / n_cols)
    size = 28 + pad
    big = -np.ones((n_rows * size + pad, n_cols * size + pad), dtype=np.float32)
    for i, img in enumerate(images):
        r, c = divmod(i, n_cols)
        big[pad + r * size: pad + r * size + 28, pad + c * size: pad + c * size + 28] = img
    return big

def clean(ax):
    # Image panels need no ticks, grid or frame.
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for side in ax.spines.values():
        side.set_visible(False)

def show_grid(images, n_cols, title, row_labels=None, save_as=None):
    # Draw a grid. Strokes are shown dark on white (display only; the data is unchanged).
    big = grid_array(images, n_cols)
    fig, ax = plt.subplots(figsize=(n_cols * 0.55, big.shape[0] / big.shape[1] * n_cols * 0.55 + 0.3))
    ax.imshow(big, cmap="gray_r", vmin=-1, vmax=1)
    ax.set_title(title, fontsize=10, loc="left")
    clean(ax)
    if row_labels is not None:
        ax.set_yticks([2 + 14 + i * 30 for i in range(len(row_labels))])
        ax.set_yticklabels(row_labels, fontsize=8)
        ax.tick_params(length=0)
    if save_as:
        fig.savefig(save_as)
    plt.show()

# 16 random training drawings per class (random, not hand-picked)
pick = torch.Generator().manual_seed(SEED)
example_rows = torch.cat([c * N_TRAIN_PER_CLASS + torch.randperm(N_TRAIN_PER_CLASS, generator=pick)[:16]
                          for c in range(len(CLASSES))])
show_grid(train_x[example_rows], 16, "Real training drawings (random sample)", CLASSES,
          f"{OUT_DIR}/figures/fig_training_examples.png")

---
## 3. Method: flow matching in plain words

**The idea.** Take a noise image $x_0$ and a real drawing $x_1$. Draw a straight line between them. At time $t$ (from 0 to 1) the point on the line is

$$x_t = (1-t)\,x_0 + t\,x_1, \qquad x_0 \sim \mathcal N(0, I),\quad x_1 \sim \text{data},\quad t \sim \text{Uniform}(0,1)$$

Moving along a straight line means the velocity never changes. It is always

$$u = \frac{dx_t}{dt} = x_1 - x_0$$

**What is learned.** A network $v_\theta(x_t, t, c)$ sees the mixed image $x_t$, the time $t$ and the class label $c$, and has to guess that velocity. The loss is a plain regression loss:

$$L(\theta) = \mathbb E\,\big\| v_\theta(x_t, t, c) - (x_1 - x_0) \big\|^2$$

**Why the guess is an average.** The network never sees $x_0$ or $x_1$, only $x_t$. Many different noise–drawing pairs pass through the same $x_t$, so the best possible guess is the *average* velocity of all the lines through that point.

*Analogy:* people cross a town square, each walking a straight line to their own exit. Standing at one spot you cannot know one person's exit, but you can learn the average walking direction at that spot and that moment. The network learns this traffic map. A new person who follows the map from a random starting point ends up at a sensible exit.

**How a drawing is generated.** Start from fresh noise and follow the learned velocity from $t=0$ to $t=1$. This is an ordinary differential equation (ODE), solved here with $N$ Euler steps of size $1/N$:

$$\frac{dx}{dt} = v_\theta(x, t, c), \qquad x_{k+1} = x_k + \tfrac{1}{N}\, v_\theta\!\left(x_k, \tfrac{k}{N}, c\right)$$

Each Euler step calls the network once, so the number of network evaluations equals $N$.

```
training:  drawing x1 --+
           noise   x0 --+--> x_t = (1-t)*x0 + t*x1 --> network v(x_t, t, class) --> compare with (x1 - x0) --> MSE loss
           time    t  --+

sampling:  noise x0 --> repeat N times:  x = x + (1/N) * v(x, t, class) --> drawing
```

**Link to probabilistic modelling**

| Idea | Where it appears here |
|---|---|
| Known probabilistic structure | The noise distribution $\mathcal N(0, I)$ and the straight-line rule are fixed by us. Nothing about them is learned. |
| Unknown component learned from data | The velocity network. About a million weights are learned, and only one best-fit value is kept rather than a full distribution over the weights. |
| Generation by sampling | Draw the root variable $x_0$ at random, then compute everything after it. All randomness enters at the root; the ODE solver itself adds none. |

The figure below shows the construction on one real training drawing.

In [ ]:
# One real drawing, one noise image, and the points on the straight line between them
x1 = train_x[example_rows[:1]].cpu()
x0 = torch.randn(x1.shape, generator=torch.Generator().manual_seed(0))
times = [0.0, 0.25, 0.5, 0.75, 1.0]

fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax, t in zip(axes, times):
    ax.imshow(((1 - t) * x0 + t * x1)[0, 0], cmap="gray_r", vmin=-2, vmax=2)
    ax.set_title(f"x_t at t = {t}", fontsize=9)
axes[5].imshow((x1 - x0)[0, 0], cmap="gray_r", vmin=-3, vmax=3)
axes[5].set_title("target x1 − x0", fontsize=9)
for ax in axes:
    clean(ax)
fig.suptitle("Straight line from noise (t = 0) to a training drawing (t = 1), and the velocity the network must predict",
             fontsize=10, x=0.01, ha="left")
fig.savefig(f"{OUT_DIR}/figures/fig_method_path.png")
plt.show()

---
## 4. The velocity network

A small U-Net: the image is shrunk 28 → 14 → 7 to see the overall shape, then grown back 7 → 14 → 28 to redraw the details. Skip connections pass the fine detail across.

The time $t$ and the class label are each turned into a vector, added together, and fed into every block. This is how the network knows *when* along the path it is and *what* it should be drawing.

- **Input:** image `x` of shape (B, 1, 28, 28), time `t` of shape (B,), label `y` of shape (B,)
- **Output:** predicted velocity, same shape as `x`

In [ ]:
def time_embedding(t, dim=64):
    # Turn each time t in [0, 1] into a vector of sines and cosines at different frequencies.
    freqs = torch.exp(-math.log(1000) * torch.arange(dim // 2, device=t.device) / (dim // 2))
    angles = 1000 * t[:, None] * freqs[None, :]
    return torch.cat([angles.sin(), angles.cos()], dim=1)


class Block(nn.Module):
    # Two 3x3 convolutions. The time+class vector is added in between.
    def __init__(self, c_in, c_out, emb_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, padding=1)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, c_out)
        self.norm2 = nn.GroupNorm(8, c_out)
        self.emb = nn.Linear(emb_dim, c_out)

    def forward(self, x, emb):
        h = F.silu(self.norm1(self.conv1(x)))
        h = h + self.emb(emb)[:, :, None, None]
        return F.silu(self.norm2(self.conv2(h)))


class VelocityNet(nn.Module):
    def __init__(self, n_classes, ch=48, emb_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(nn.Linear(64, emb_dim), nn.SiLU(), nn.Linear(emb_dim, emb_dim))
        self.class_emb = nn.Embedding(n_classes, emb_dim)
        self.down1 = Block(1, ch, emb_dim)                      # 28 x 28
        self.down2 = Block(ch, ch * 2, emb_dim)                 # 14 x 14
        self.mid = Block(ch * 2, ch * 4, emb_dim)               #  7 x  7
        self.up2 = Block(ch * 4 + ch * 2, ch * 2, emb_dim)      # 14 x 14
        self.up1 = Block(ch * 2 + ch, ch, emb_dim)              # 28 x 28
        self.out = nn.Conv2d(ch, 1, 1)

    def forward(self, x, t, y):
        emb = self.time_mlp(time_embedding(t)) + self.class_emb(y)
        d1 = self.down1(x, emb)
        d2 = self.down2(F.avg_pool2d(d1, 2), emb)
        m = self.mid(F.avg_pool2d(d2, 2), emb)
        u2 = self.up2(torch.cat([F.interpolate(m, scale_factor=2), d2], dim=1), emb)
        u1 = self.up1(torch.cat([F.interpolate(u2, scale_factor=2), d1], dim=1), emb)
        return self.out(u1)


n_params = sum(p.numel() for p in VelocityNet(len(CLASSES)).parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

---
## 5. The three key functions

| Function | What it does | Inputs | Outputs |
|---|---|---|---|
| `make_training_example` | Builds what the network sees and what it should answer | `x1`: a batch of real drawings | `xt`: noise–drawing mix, `t`: the random times, `target`: the velocity `x1 − x0` |
| `train_step` | One learning step: predict, measure the error, adjust the weights | the model, the optimiser, drawings `x1`, labels `y` | the loss value for this batch |
| `generate` | Turns noise into drawings by following the learned velocity with Euler steps | the model, starting `noise`, `labels`, number of steps | the generated drawings (and the in-between states if asked) |

In [ ]:
def make_training_example(x1):
    # x1: real drawings, shape (B, 1, 28, 28), values in [-1, 1]
    x0 = torch.randn_like(x1)                        # Gaussian noise: the start of each line
    t = torch.rand(x1.shape[0], device=x1.device)    # one random time per drawing
    t_img = t[:, None, None, None]                   # reshaped so it multiplies whole images
    xt = (1 - t_img) * x0 + t_img * x1               # point on the straight line at time t
    target = x1 - x0                                 # velocity of that line
    return xt, t, target


def train_step(model, optimiser, x1, y):
    xt, t, target = make_training_example(x1)
    prediction = model(xt, t, y)                     # the network's guess of the velocity
    loss = F.mse_loss(prediction, target)            # average squared error
    optimiser.zero_grad()                            # clear the old gradients
    loss.backward()                                  # how should each weight change?
    optimiser.step()                                 # change the weights a little
    return loss.item()


@torch.no_grad()
def generate(model, noise, labels, n_steps, return_path=False):
    # Euler integration of dx/dt = v(x, t, class) from t = 0 to t = 1.
    # One network call per step, so network evaluations = n_steps.
    x = noise.to(device).clone()
    y = labels.to(device)
    dt = 1.0 / n_steps
    path = [x.cpu()]
    for k in range(n_steps):
        t = torch.full((x.shape[0],), k * dt, device=device)
        x = x + dt * model(x, t, y)
        if return_path:
            path.append(x.cpu())
    x = x.clamp(-1, 1).cpu()                         # keep pixel values in the valid range
    return (x, path) if return_path else x

**Quick checks before training** (on one batch and an untrained network):

1. The shapes are what the table above says.
2. Starting from `xt` and moving with the target velocity for the remaining time `1 − t` lands exactly on the real drawing. This confirms `xt` and `target` belong to the same straight line.
3. After one backward pass, every weight in the network has a gradient, so every part of it is being trained.

In [ ]:
torch.manual_seed(SEED)
check_model = VelocityNet(len(CLASSES)).to(device)
xb, yb = train_x[:8], train_y[:8]
xt, t, target = make_training_example(xb)
print("xt", tuple(xt.shape), "| t", tuple(t.shape), "| target", tuple(target.shape),
      "| model output", tuple(check_model(xt, t, yb).shape))

lands_on_drawing = torch.allclose(xt + (1 - t[:, None, None, None]) * target, xb, atol=1e-5)
print("xt + (1 - t) * target equals the drawing:", lands_on_drawing)

F.mse_loss(check_model(xt, t, yb), target).backward()
no_gradient = [name for name, p in check_model.named_parameters() if p.grad is None or p.grad.abs().sum() == 0]
print("Weights without a gradient:", no_gradient if no_gradient else "none")
del check_model

---
## 6. Train the baseline

Each step picks 128 random training drawings, builds the training example and updates the weights once. The weights are saved at four points during training, so the checkpoints can be compared later without training separate models.

In [ ]:
def weights_path(step):
    return f"{OUT_DIR}/weights/velocity_step{step}.pt"

if RUN_TRAINING:
    torch.manual_seed(SEED)
    model = VelocityNet(len(CLASSES)).to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    losses, checkpoint_seconds = [], {}
    start = time.time()
    for step in range(1, TOTAL_STEPS + 1):
        batch = torch.randint(0, len(train_x), (BATCH_SIZE,), device=device)
        losses.append(train_step(model, optimiser, train_x[batch], train_y[batch]))
        if step in CHECKPOINT_STEPS:
            torch.save(model.state_dict(), weights_path(step))
            checkpoint_seconds[step] = round(time.time() - start, 1)
            print(f"step {step:5d} | mean loss of last 100 steps {np.mean(losses[-100:]):.4f} | {checkpoint_seconds[step]:.0f} s")
    pd.DataFrame({"step": range(1, TOTAL_STEPS + 1), "loss": losses}).to_csv(f"{OUT_DIR}/training_loss.csv", index=False)
    json.dump({str(k): v for k, v in checkpoint_seconds.items()}, open(f"{OUT_DIR}/training_time.json", "w"))
else:
    losses = pd.read_csv(f"{OUT_DIR}/training_loss.csv")["loss"].tolist()
    checkpoint_seconds = {int(k): v for k, v in json.load(open(f"{OUT_DIR}/training_time.json")).items()}
    print("Training skipped. Using saved weights and the saved loss log.")

def load_model(step):
    m = VelocityNet(len(CLASSES)).to(device)
    m.load_state_dict(torch.load(weights_path(step), map_location=device))
    return m.eval()

models = {step: load_model(step) for step in CHECKPOINT_STEPS}
FINAL_STEP = CHECKPOINT_STEPS[-1]

In [ ]:
smooth = pd.Series(losses).rolling(100).mean()
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(range(1, len(losses) + 1), losses, color="#c9c8c2", linewidth=0.6, label="loss of each batch")
ax.plot(range(1, len(losses) + 1), smooth, color=BLUE, label="average of the last 100 batches")
for step in CHECKPOINT_STEPS:
    ax.axvline(step, color=GREY, linestyle=":", linewidth=1)
    ax.text(step, ax.get_ylim()[1], f" step {step}", fontsize=8, color=GREY, va="top", ha="right", rotation=90)
ax.set_xlabel("training step"); ax.set_ylabel("MSE loss"); ax.set_title("Training loss", loc="left", fontsize=10)
ax.legend(frameon=False)
fig.savefig(f"{OUT_DIR}/figures/fig_training_loss.png")
plt.show()

The loss does not go to zero and is not expected to. The network cannot know which exact noise image and drawing produced an `xt`, so some error always remains. This is one reason the loss is not used to judge image quality below.

---
## 7. How the outputs are scored

**Recognisability score.** A separate small classifier is trained on the real training drawings to tell the three categories apart. The score of a set of generated images is the share that the classifier assigns to the class that was asked for. Its accuracy on the held-out real drawings is the reference: generated drawings cannot be expected to beat real ones.

**Grey-pixel share.** A real doodle is almost all paper or ink, with only thin grey edges. This measure is the share of pixels that are neither (value between −0.5 and 0.5). A blurry or smudged image has more grey than a real drawing, so a value near the real one is better.

**Diversity.** The average pixel distance between two different images of the same class.

For all three measures, the same number computed on held-out real drawings is the reference.

**Limits of these measures**

- The classifier only knows three categories. A messy blob still gets one of the three labels, and chance level is 33%.
- It has only seen clean real drawings, so its answers on noisy or half-formed images are less reliable.
- A high score means "looks like the right category to this classifier". It does not mean the drawing is good, new or varied. A blurry image can still be recognised, which is why the grey-pixel share is reported next to it.
- The grey-pixel share only looks at pixel values, not at shapes. A sharp scribble scores as well as a sharp cat.
- The diversity number goes up when images are noisy and down when they are blurry, so it has to be read together with the grids.

In [ ]:
class SmallClassifier(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc = nn.Linear(32 * 7 * 7, n_classes)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)      # 28 -> 14
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)      # 14 -> 7
        return self.fc(x.flatten(1))


torch.manual_seed(SEED)
classifier = SmallClassifier(len(CLASSES)).to(device)
clf_optimiser = torch.optim.Adam(classifier.parameters(), lr=1e-3)
for epoch in range(5):
    order = torch.randperm(len(train_x), device=device)
    for i in range(0, len(order), 128):
        rows = order[i:i + 128]
        clf_loss = F.cross_entropy(classifier(train_x[rows]), train_y[rows])
        clf_optimiser.zero_grad(); clf_loss.backward(); clf_optimiser.step()
classifier.eval()


@torch.no_grad()
def predict_class(images):
    return classifier(images.to(device)).argmax(1).cpu()

def recognisability(images, labels):
    # Share of images the classifier assigns to the requested class: overall, then per class.
    labels = labels.cpu()
    correct = (predict_class(images) == labels).float()
    return correct.mean().item(), [correct[labels == c].mean().item() for c in range(len(CLASSES))]

def grey_pixel_share(images):
    # Share of pixels that are neither clearly paper (-1) nor clearly ink (+1). Higher = blurrier.
    return (images.abs() < 0.5).float().mean().item()

def diversity(images, labels):
    # Average distance between two different images of the same class. Larger = more varied.
    images, labels = images.cpu(), labels.cpu()
    values = []
    for c in range(len(CLASSES)):
        flat = images[labels == c].flatten(1)
        n = len(flat)
        values.append(torch.cdist(flat, flat).sum().item() / (n * (n - 1)))
    return float(np.mean(values))


# Reference values from real held-out drawings (the first 100 per class, to match the generated set size)
ref_rows = torch.cat([c * (N_PER_CLASS - N_TRAIN_PER_CLASS) + torch.arange(len(EVAL_SEEDS)) for c in range(len(CLASSES))])
real_score, real_per_class = recognisability(test_x, test_y)
real_grey = grey_pixel_share(test_x[ref_rows])
real_diversity = diversity(test_x[ref_rows], test_y[ref_rows])
print(f"Classifier accuracy on all {len(test_x):,} held-out real drawings: {real_score:.3f}")
print("  per class:", {name: round(v, 3) for name, v in zip(CLASSES, real_per_class)})
print(f"Real held-out drawings (100 per class): grey-pixel share {real_grey:.3f}, diversity {real_diversity:.2f}")

---
## 8. Baseline samples

**Baseline = final checkpoint, Euler sampler, 50 steps.**

**Fixed inputs for every configuration.** Seeds 0 to 99 give 100 fixed noise images. Each noise image is used once per class, so every configuration is scored on the same 300 inputs (100 per class). The grids show the first 16 seeds per class. Nothing is selected after looking at the results.

In [ ]:
def make_noise(seeds):
    # One fixed noise image per seed. Made on the CPU so the same seed gives the same noise on any machine.
    return torch.stack([torch.randn(1, 28, 28, generator=torch.Generator().manual_seed(s)) for s in seeds])

EVAL_NOISE = make_noise(EVAL_SEEDS).repeat(len(CLASSES), 1, 1, 1)                # same 100 noise images for each class
EVAL_LABELS = torch.arange(len(CLASSES)).repeat_interleave(len(EVAL_SEEDS))
GRID_ROWS = torch.cat([c * len(EVAL_SEEDS) + torch.arange(N_GRID) for c in range(len(CLASSES))])

generate(models[FINAL_STEP], EVAL_NOISE[:8], EVAL_LABELS[:8], 2)                  # warm-up so timings are fair

def evaluate(name, step, n_steps):
    # Generate the full fixed set for one configuration, score it and save its grid.
    start = time.time()
    images = generate(models[step], EVAL_NOISE, EVAL_LABELS, n_steps)
    seconds = time.time() - start
    overall, per_class = recognisability(images, EVAL_LABELS)
    row = {"configuration": name, "checkpoint_step": step, "euler_steps": n_steps, "network_evaluations": n_steps,
           "n_images": len(images), "sampling_seconds": round(seconds, 2), "recognisability": round(overall, 3)}
    row.update({f"recognisability_{c}": round(v, 3) for c, v in zip(CLASSES, per_class)})
    row["grey_pixel_share"] = round(grey_pixel_share(images), 3)
    row["diversity"] = round(diversity(images, EVAL_LABELS), 2)
    show_grid(images[GRID_ROWS], N_GRID, f"{name}   |   recognisability {overall:.2f}", CLASSES,
              f"{OUT_DIR}/grids/grid_{name}.png")
    return row, images

baseline_row, baseline_images = evaluate(f"step{FINAL_STEP}_euler{BASE_ODE_STEPS}", FINAL_STEP, BASE_ODE_STEPS)
pd.DataFrame([baseline_row])

**Is the class label obeyed?** Rows are the class that was requested, columns are what the classifier saw, counted over the 300 baseline images.

In [ ]:
confusion = pd.crosstab(pd.Series([CLASSES[i] for i in EVAL_LABELS], name="requested"),
                        pd.Series([CLASSES[i] for i in predict_class(baseline_images)], name="classifier says"))
confusion = confusion.reindex(index=CLASSES, columns=CLASSES, fill_value=0)
confusion.to_csv(f"{OUT_DIR}/baseline_confusion.csv")
confusion

**From noise to drawing.** The same sampler, with the in-between states kept. Two seeds per class.

In [ ]:
path_rows = torch.cat([c * len(EVAL_SEEDS) + torch.arange(2) for c in range(len(CLASSES))])
_, path = generate(models[FINAL_STEP], EVAL_NOISE[path_rows], EVAL_LABELS[path_rows], BASE_ODE_STEPS, return_path=True)
shown_steps = [BASE_ODE_STEPS * i // 5 for i in range(6)]      # t = 0, 0.2, ..., 1

fig, axes = plt.subplots(len(path_rows), len(shown_steps), figsize=(len(shown_steps) * 1.1, len(path_rows) * 1.1))
for r in range(len(path_rows)):
    for c, k in enumerate(shown_steps):
        axes[r, c].imshow(path[k][r, 0], cmap="gray_r", vmin=-2, vmax=2)
        clean(axes[r, c])
        if r == 0:
            axes[r, c].set_title(f"t = {k / BASE_ODE_STEPS:.1f}", fontsize=9)
    axes[r, 0].set_ylabel(CLASSES[EVAL_LABELS[path_rows[r]]], fontsize=8)
fig.suptitle("States along the ODE, from noise to drawing", fontsize=10, x=0.02, y=0.96, ha="left")
fig.savefig(f"{OUT_DIR}/figures/fig_noise_to_drawing.png")
plt.show()

**Where does the randomness come from?** The Euler solver is a fixed recipe: the same starting noise always gives the same drawing. A different drawing needs a different starting noise. The check below confirms this.

In [ ]:
label = torch.tensor([0])
run_a = generate(models[FINAL_STEP], make_noise([0]), label, BASE_ODE_STEPS)
run_b = generate(models[FINAL_STEP], make_noise([0]), label, BASE_ODE_STEPS)
run_c = generate(models[FINAL_STEP], make_noise([1]), label, BASE_ODE_STEPS)
print("Same seed, run twice  -> largest pixel difference:", (run_a - run_b).abs().max().item())
print("Different seed        -> largest pixel difference:", (run_a - run_c).abs().max().item())

---
## 9. Expectation before the comparisons

- **Euler steps:** recognisability should rise as the number of Euler steps goes up, because smaller steps follow the learned path more closely.

Section 15 checks this against the results.

---
## 10. Comparison 1 (training factor): earlier and later checkpoints

**Only the checkpoint changes.** The sampler (Euler, 50 steps), the noise images, the labels and the classifier are the same in every row. The cost column is the training time needed to reach each checkpoint.

In [ ]:
checkpoint_rows, checkpoint_images = [], {}
for step in CHECKPOINT_STEPS:
    if step == FINAL_STEP:
        row, images = dict(baseline_row), baseline_images      # the baseline, already generated above
    else:
        row, images = evaluate(f"step{step}_euler{BASE_ODE_STEPS}", step, BASE_ODE_STEPS)
    row["training_seconds"] = checkpoint_seconds[step]
    checkpoint_rows.append(row); checkpoint_images[step] = images

results_checkpoints = pd.DataFrame(checkpoint_rows)
results_checkpoints.to_csv(f"{OUT_DIR}/results_checkpoints.csv", index=False)
results_checkpoints

---
## 11. Comparison 2 (inference factor): number of Euler steps

**Only the number of Euler steps changes.** The model (final checkpoint), the noise images, the labels and the classifier are the same in every row. With Euler, network evaluations = steps, so the sampling time should grow roughly in line with the step count.

In [ ]:
ode_rows, ode_images = [], {}
for n_steps in ODE_STEPS_LIST:
    if n_steps == BASE_ODE_STEPS:
        row, images = dict(baseline_row), baseline_images      # the baseline, already generated above
    else:
        row, images = evaluate(f"step{FINAL_STEP}_euler{n_steps}", FINAL_STEP, n_steps)
    ode_rows.append(row); ode_images[n_steps] = images

results_ode = pd.DataFrame(ode_rows)
results_ode.to_csv(f"{OUT_DIR}/results_ode_steps.csv", index=False)
results_ode

---
## 12. Matched comparison figures

The same eight seeds per class under every setting, so each column can be followed from one setting to the next. These are the first eight seeds, not chosen for looks. The full 16-per-class grids are in `partC_outputs/grids/`.

In [ ]:
def matched_figure(image_sets, titles, save_as, n_show=8):
    # One panel per configuration, same seeds in the same positions.
    rows = torch.cat([c * len(EVAL_SEEDS) + torch.arange(n_show) for c in range(len(CLASSES))])
    fig, axes = plt.subplots(1, len(image_sets), figsize=(len(image_sets) * 3.1, 1.6))
    for ax, images, title in zip(axes, image_sets, titles):
        ax.imshow(grid_array(images[rows], n_show), cmap="gray_r", vmin=-1, vmax=1)
        ax.set_title(title, fontsize=9); clean(ax)
    axes[0].set_yticks([2 + 14 + i * 30 for i in range(len(CLASSES))]); axes[0].set_yticklabels(CLASSES, fontsize=8)
    axes[0].tick_params(length=0)
    fig.savefig(save_as)
    plt.show()

matched_figure([checkpoint_images[s] for s in CHECKPOINT_STEPS], [f"training step {s}" for s in CHECKPOINT_STEPS],
               f"{OUT_DIR}/figures/fig_compare_checkpoints.png")
matched_figure([ode_images[n] for n in ODE_STEPS_LIST], [f"{n} Euler steps" for n in ODE_STEPS_LIST],
               f"{OUT_DIR}/figures/fig_compare_ode_steps.png")

In [ ]:
def comparison_chart(columns, labels, colours, y_label, reference, reference_label, save_as, chance=None):
    # Left panel: checkpoints. Right panel: Euler steps. Same y-axis in both.
    fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
    panels = [(axes[0], results_checkpoints, "checkpoint_step", "training step (log scale)", "Comparison 1: checkpoints"),
              (axes[1], results_ode, "euler_steps", "Euler steps = network evaluations (log scale)", "Comparison 2: Euler steps")]
    for ax, table, x_col, x_label, title in panels:
        for column, label, colour in zip(columns, labels, colours):
            ax.plot(table[x_col], table[column], marker="o", color=colour, label=label)
        ax.axhline(reference, color=GREY, linestyle="--", linewidth=1, label=reference_label)
        if chance is not None:
            ax.axhline(chance, color=GREY, linestyle=":", linewidth=1, label="chance")
        ax.set_xscale("log"); ax.set_xticks(table[x_col]); ax.set_xticklabels(table[x_col]); ax.minorticks_off()
        ax.set_xlabel(x_label); ax.set_title(title, loc="left", fontsize=10); ax.set_ylim(bottom=0)
    axes[0].set_ylabel(y_label)
    axes[1].legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1.0, 0.5))
    fig.savefig(save_as)
    plt.show()

comparison_chart([f"recognisability_{c}" for c in CLASSES], CLASSES, CLASS_COLOURS, "recognisability score",
                 real_score, "real held-out drawings", f"{OUT_DIR}/figures/fig_recognisability.png", chance=1 / len(CLASSES))
comparison_chart(["grey_pixel_share"], ["generated"], [BLUE], "grey-pixel share (lower = sharper)",
                 real_grey, "real held-out drawings", f"{OUT_DIR}/figures/fig_grey_pixel_share.png")

---
## 13. Is the model copying training drawings?

For each baseline image, the closest training drawing is found by pixel distance. Looking alike is not proof of copying, so a yardstick is needed: **how close is a genuinely new drawing to its nearest training drawing?** The held-out real drawings answer that, because the model has never seen them.

- If generated images sit about as far from the training set as held-out drawings do, there is no sign of copying.
- If they sit much closer, that would suggest memorisation.

*Limit:* pixel distance is a blunt tool. A copy shifted by two pixels looks far away, and two different drawings with thin strokes in similar places can look close.

In [ ]:
train_flat = train_x.flatten(1)

def nearest_training(images):
    # For each image: distance to, and row number of, the closest training drawing.
    distances = torch.cdist(images.to(device).flatten(1), train_flat)
    best = distances.min(dim=1)
    return best.values.cpu(), best.indices.cpu()

gen_dist, gen_idx = nearest_training(baseline_images)
real_dist, _ = nearest_training(test_x[ref_rows])

summary = pd.DataFrame({"generated (baseline, 300 images)": gen_dist.numpy(),
                        "real held-out (300 images)": real_dist.numpy()}).describe().loc[["min", "25%", "50%", "75%", "max"]].round(2)
summary.to_csv(f"{OUT_DIR}/nearest_training_distances.csv")
summary

In [ ]:
bins = np.linspace(0, max(gen_dist.max(), real_dist.max()).item(), 30)
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(real_dist.numpy(), bins=bins, color=ORANGE, alpha=0.6, label="real held-out drawings")
ax.hist(gen_dist.numpy(), bins=bins, color=BLUE, alpha=0.6, label="generated (baseline)")
ax.set_xlabel("pixel distance to the nearest training drawing"); ax.set_ylabel("number of images")
ax.set_title("How close are images to the training set?", loc="left", fontsize=10)
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))
fig.savefig(f"{OUT_DIR}/figures/fig_nearest_distance_hist.png")
plt.show()

# First 8 seeds per class: generated image on top, its nearest training drawing underneath
show = torch.cat([c * len(EVAL_SEEDS) + torch.arange(8) for c in range(len(CLASSES))])
fig, axes = plt.subplots(2, len(show), figsize=(len(show) * 0.7, 1.9))
for j, row in enumerate(show):
    axes[0, j].imshow(baseline_images[row, 0], cmap="gray_r", vmin=-1, vmax=1)
    axes[1, j].imshow(train_x[gen_idx[row], 0].cpu(), cmap="gray_r", vmin=-1, vmax=1)
    axes[0, j].set_title(f"{gen_dist[row]:.1f}", fontsize=7)
    for ax in axes[:, j]:
        clean(ax)
axes[0, 0].set_ylabel("generated", fontsize=7); axes[1, 0].set_ylabel("nearest\ntraining", fontsize=7)
fig.suptitle("Generated images (top) and their closest training drawings (bottom). Number = pixel distance.",
             fontsize=9, x=0.01, y=1.12, ha="left")
fig.savefig(f"{OUT_DIR}/figures/fig_nearest_training_pairs.png")
plt.show()

---
## 14. Save the run details and download everything

In [ ]:
run_info = {
    "classes": CLASSES, "seed": SEED,
    "train_drawings": len(train_x), "held_out_drawings": len(test_x),
    "image_size": "28x28 grayscale, scaled to [-1, 1], no augmentation",
    "model": "small U-Net velocity network, class-conditional", "trainable_parameters": n_params,
    "optimiser": "Adam", "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE,
    "total_steps": TOTAL_STEPS, "epochs_equivalent": round(TOTAL_STEPS * BATCH_SIZE / len(train_x), 1),
    "checkpoint_steps": CHECKPOINT_STEPS, "training_seconds_at_checkpoint": checkpoint_seconds,
    "baseline_sampler": f"Euler, {BASE_ODE_STEPS} steps", "ode_steps_compared": ODE_STEPS_LIST,
    "eval_seeds": f"{EVAL_SEEDS[0]} to {EVAL_SEEDS[-1]} (same noise reused for every class and configuration)",
    "classifier_accuracy_on_held_out": round(real_score, 3),
    "real_grey_pixel_share": round(real_grey, 3), "real_diversity": round(real_diversity, 2),
    "hardware": hardware, "device": device, "versions": versions,
}
json.dump(run_info, open(f"{OUT_DIR}/run_info.json", "w"), indent=2)
print(json.dumps(run_info, indent=2))

shutil.make_archive("partC_outputs", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("partC_outputs.zip")
except ImportError:
    print("Not on Colab: partC_outputs.zip is saved next to the notebook.")

---
## 15. What I found

The numbers below are from my Colab run on a Tesla T4. The tables above show the values for the current run.

- **More Euler steps did not always help.** The expectation in section 9 held for cats (0.85 at 2 steps, 0.96 at 50) but failed for fish (1.00 at 2 steps, 0.88 at 50). With very few steps the model draws a blurry "average" fish that the classifier finds easy. With more steps the drawings are sharper and more varied, so some are harder to classify. The 50-step fish score is closer to the score for real held-out fish (0.928), so a higher score is not always a better drawing.
- **No sign of copying.** The nearest-distance check in section 13 found no evidence that the model copies training drawings.
- **Limits.** The scores depend on a small classifier that is not perfect, and pixel distance is a blunt check for copying. The model only makes 28 × 28 doodles for three classes.